## V-Jepa2.1


*  VideoMAE/V-JEPA: temporal kernel = 2 (each patch spans 2 frames, captures short motion)
  

### PatchEmbeddings

* So far, we did a 2D patch embedding, now we need to form the intution for adding a temporal dimension.  

Before we had:

```
# 2D: Image → patches
# Input:  (B, 3, 224, 224)  — one frame
# Conv2d: kernel=16x16, stride=16
# Output: (B, embed_dim, 14, 14) → flatten → (B, 196, embed_dim)
```

* Now, each token, instead of a 2D patch of an image is a 3D cude of the video. 
* If we have 16 frames as input, and we use a temporal kernel of 2 (meaning each patch spans 2 frames), then we have 8 timesteps. 
* Total tokens is now 8 (temporal chunks) * 14 (patches down) * 14 (patches up) =  1568 tokens or unique 3d patches. 
* Each token/patch captures 2 consecutive frames, 16x16 pixels (if we resize to 224x224) and 3 color channels. 
* Transformer mechanism then uses attention to let all 1,568 tokens attend to each other. 
* We go from 196 patches/tokens to 1,568! 

In [ ]:
import torch
import torch.nn as nn

# A tiny video: 16 frames, 224x224, RGB
video = torch.randn(1, 3, 16, 224, 224)  # (B, C, T, H, W)

# 3D patch embedding
# kernel = (2, 16, 16) → 2 frames in time, 16x16 in space
# stride = (2, 16, 16) → non-overlapping in all dimensions
patch_embed = nn.Conv3d(
    in_channels=3,
    out_channels=768,         # embed_dim
    kernel_size=(2, 16, 16),  # (temporal, height, width)
    stride=(2, 16, 16),       # no overlap
)
#  nn.Conv2d(in_channels=3,
    # out_channels = 768,            <- Frame Vit
    # kernel_size = (16, 16),
    # stride = (16, 16))

patches = patch_embed(video)
print(f"Input video:  {video.shape}")       # (1, 3, 16, 224, 224)
print(f"After Conv3D: {patches.shape}")      # (1, 768, 8, 14, 14)

# Flatten spatial+temporal into a sequence
B, D, T, H, W = patches.shape
tokens = patches.reshape(B, D, -1).transpose(1, 2)
print(f"Token sequence: {tokens.shape}")     # (1, 1568, 768)
# 1568 = 8 (time) × 14 (height) × 14 (width) patches

Input video:  torch.Size([1, 3, 16, 224, 224])
After Conv3D: torch.Size([1, 768, 8, 14, 14])
Token sequence: torch.Size([1, 1568, 768])
